# Stage 11 — Naked option buying scale-up (218 stocks + 8 indices, 1H)

Same entry signal as the debit spread strategy (`ema_adx_signal.py`, EMA 9/21 + rising ADX, 1H), but `option_buying_trades.py` exits with a trailing stop instead of a fixed 1.5R target - naked buying has no structural cap the way a spread does, so a fixed target would throw away the whole reason to skip the hedge leg.

**4H trend hard filter added after validation** (`apply_4h_trend_filter()`, baked into `option_buying_scale_pipeline.process_instrument()`/`reprocess_instrument()`): only 1H entries that agree with the symbol's own 4H Supertrend direction at that moment survive. Tested on 12,608 real closed trades matched to their 4H trend state: aligned entries won 53.3% vs 47.2% against, mean R 0.166 vs 0.024, median R +0.08 vs -0.46 - the typical against-trend trade was a loser, the typical aligned trade wasn't. This roughly halves signal frequency by construction, so per-instrument trade counts here will be noticeably lower than a pre-filter run.

A sandbox run (cached 1H data, hand-rolled pandas_ta stand-in since the real library isn't installable there) with the filter applied gave 153 qualified instruments (>=15 closed trades, down from 189 pre-filter), 70.6% positive avg R, mean 0.198, median 0.170 - all higher than the pre-filter numbers, at the cost of a thinner per-instrument sample (mean ~19 closed trades vs ~30+ before). That run used whatever history happened to already be cached per symbol (inconsistent windows), not a clean uniform one. **This notebook does the real, clean version**: same `from_date=2023-01-01` window as `08_debit_spread_scale_all_fno.ipynb` (Stage 9), `force_refresh=True` so every symbol gets the same window regardless of what's currently cached.

New: `src/option_buying_scale_pipeline.py` - parallel to `debit_spread_scale_pipeline.py`, not a modification. Same resume/failure-isolation/rate-limit discipline.

In [1]:
import sys
sys.path.append("..")

from datetime import datetime
from src.auth import get_kite
from src.instruments import get_fno_stock_list, build_nse_token_lookup, get_index_token
from src.option_buying_scale_pipeline import run_all_instruments, build_comparison, save_comparison

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)
stock_list = get_fno_stock_list(kite)
len(stock_list)

213

In [2]:
results = run_all_instruments(
    kite, token_lookup, stock_list,
    from_date=datetime(2023, 1, 1),
    to_date=datetime.now(),
    force_refresh=True
)
results

[1/213] 360ONE — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-08-07...
Saved 6197 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/360ONE_60minute.csv
Saved 41 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/option_buying/360ONE_trades.csv
[1/213] 360ONE — done (trades=41, win_rate=43.9, avg_r=0.1)
[2/213] ABB — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-08-07...
Saved 6197 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/ABB_60minute.csv
Saved 33 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/option_buyin

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,pct_trades_trailing_activated,symbol,status,error
0,41.0,41.0,0.0,43.9,0.10,-0.81,"{'STOP': 20, 'TRAIL_STOP': 11, 'TIME': 10}",7.0,41.5,360ONE,SUCCESS,
1,33.0,33.0,0.0,48.5,-0.07,-1.00,"{'STOP': 17, 'TRAIL_STOP': 12, 'TIME': 4}",6.0,45.5,ABB,SUCCESS,
2,42.0,42.0,0.0,50.0,0.02,-0.04,"{'STOP': 20, 'TRAIL_STOP': 19, 'TIME': 3}",6.0,47.6,ABCAPITAL,SUCCESS,
3,33.0,33.0,0.0,63.6,0.49,0.40,"{'TRAIL_STOP': 17, 'STOP': 11, 'TIME': 5}",5.0,63.6,ADANIENSOL,SUCCESS,
4,37.0,37.0,0.0,45.9,-0.02,-1.00,"{'STOP': 20, 'TRAIL_STOP': 12, 'TIME': 5}",4.0,45.9,ADANIENT,SUCCESS,
...,...,...,...,...,...,...,...,...,...,...,...,...
208,28.0,28.0,0.0,57.1,0.48,0.17,"{'TRAIL_STOP': 12, 'STOP': 11, 'TIME': 5}",9.0,57.1,VOLTAS,SUCCESS,
209,18.0,18.0,0.0,61.1,0.24,0.21,"{'TRAIL_STOP': 9, 'STOP': 6, 'TIME': 3}",6.0,55.6,WAAREEENER,SUCCESS,
210,40.0,39.0,1.0,59.0,0.20,0.18,"{'TRAIL_STOP': 20, 'STOP': 16, 'TIME': 3}",7.0,56.4,WIPRO,SUCCESS,
211,48.0,48.0,0.0,41.7,0.12,-1.00,"{'STOP': 28, 'TRAIL_STOP': 18, 'TIME': 2}",5.5,41.7,YESBANK,SUCCESS,


## Indices — same 6 as the debit spread run, separate token lookup

In [3]:
from src.option_buying_scale_pipeline import process_instrument
import pandas as pd

indices = [
    {"symbol": "NIFTY", "tradingsymbol": "NIFTY 50", "exchange": "NSE"},
    {"symbol": "BANKNIFTY", "tradingsymbol": "NIFTY BANK", "exchange": "NSE"},
    {"symbol": "SENSEX", "tradingsymbol": "SENSEX", "exchange": "BSE"},
    {"symbol": "NIFTYNXT50", "tradingsymbol": "NIFTY NEXT 50", "exchange": "NSE"},
    {"symbol": "MIDCPNIFTY", "tradingsymbol": "NIFTY MID SELECT", "exchange": "NSE"},
    {"symbol": "FINNIFTY", "tradingsymbol": "NIFTY FIN SERVICE", "exchange": "NSE"},
]

index_results = []
for idx in indices:
    token = get_index_token(kite, idx["tradingsymbol"], idx["exchange"])
    summary = process_instrument(idx["symbol"], token, kite,
                                  from_date=datetime(2023, 1, 1), to_date=datetime.now())
    summary["status"] = "SUCCESS"
    summary["error"] = ""
    index_results.append(summary)
    print(f"{idx['symbol']} — refreshed")

index_results_df = pd.DataFrame(index_results)
index_results_df

Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-08-07...
Saved 6201 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/NIFTY_60minute.csv
Saved 32 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/option_buying/NIFTY_trades.csv
NIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-08-07...
Saved 6201 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/BANKNIFTY_60minute.csv
Saved 33 trades to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/option_buying/BANKNIFTY_trades.csv
BANKNIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-0

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,pct_trades_trailing_activated,symbol,status,error
0,32,32,0,46.9,0.16,-1.00,"{'STOP': 17, 'TRAIL_STOP': 8, 'TIME': 7}",6.5,40.6,NIFTY,SUCCESS,
1,33,33,0,54.5,0.22,0.25,"{'TRAIL_STOP': 14, 'STOP': 14, 'TIME': 5}",7.0,54.5,BANKNIFTY,SUCCESS,
2,27,27,0,55.6,0.20,0.15,"{'TRAIL_STOP': 11, 'STOP': 10, 'TIME': 6}",7.0,48.1,SENSEX,SUCCESS,
3,35,35,0,60.0,0.36,0.13,"{'TRAIL_STOP': 16, 'STOP': 13, 'TIME': 6}",7.0,60.0,NIFTYNXT50,SUCCESS,
4,41,41,0,53.7,0.33,0.12,"{'STOP': 17, 'TRAIL_STOP': 16, 'TIME': 8}",5.0,51.2,MIDCPNIFTY,SUCCESS,
5,36,36,0,41.7,-0.11,-1.00,"{'STOP': 19, 'TRAIL_STOP': 12, 'TIME': 5}",6.0,41.7,FINNIFTY,SUCCESS,


## Aggregate comparison — the actual verdict

In [4]:
all_results = pd.concat([results, index_results_df], ignore_index=True)
comparison = build_comparison(all_results)
save_comparison(comparison)
comparison

Saved option buying comparison of 214 instruments to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/OPTION_BUYING_COMPARISON.xlsx


,symbol,total_trades,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held,pct_trades_trailing_activated
0,SWIGGY,18.0,18.0,83.3,1.21,0.94,6.0,77.8
1,HINDUNILVR,37.0,37.0,64.9,0.86,0.77,9.0,56.8
2,SHRIRAMFIN,38.0,38.0,68.4,0.81,0.20,7.0,63.2
3,RVNL,42.0,42.0,73.8,0.79,0.68,7.0,71.4
4,SUZLON,37.0,37.0,59.5,0.72,0.48,6.0,59.5
...,...,...,...,...,...,...,...,...
209,NTPC,31.0,31.0,51.6,-0.23,0.06,6.0,45.2
210,COALINDIA,40.0,40.0,42.5,-0.26,-1.00,6.0,40.0
211,POLICYBZR,39.0,39.0,30.8,-0.27,-1.00,5.0,25.6
212,AMBUJACEM,29.0,29.0,44.8,-0.31,-1.00,6.0,41.4


In [5]:
# Same >= 15 closed trades floor used everywhere else in this project.
enough_trades = comparison[comparison["closed_trades"] >= 15]

print(f"Instruments processed: {len(comparison)}")
print(f"Instruments with >= 15 closed trades: {len(enough_trades)}")
print(f"Fraction with positive avg_r_achieved: "
      f"{(enough_trades['avg_r_achieved'] > 0).mean() * 100:.1f}%")
print()
print("avg_r_achieved distribution:")
print(enough_trades["avg_r_achieved"].describe())
print()
print("win_rate_pct distribution:")
print(enough_trades["win_rate_pct"].describe())
print()
print("pct_trades_trailing_activated distribution:")
print(enough_trades["pct_trades_trailing_activated"].describe())

Instruments processed: 214
Instruments with >= 15 closed trades: 214
Fraction with positive avg_r_achieved: 79.9%

avg_r_achieved distribution:
count    214.000000
mean       0.192757
std        0.241164
min       -0.330000
25%        0.040000
50%        0.170000
75%        0.317500
max        1.210000
Name: avg_r_achieved, dtype: float64

win_rate_pct distribution:
count    214.000000
mean      54.358879
std        8.231366
min       30.800000
25%       48.600000
50%       54.500000
75%       59.500000
max       83.300000
Name: win_rate_pct, dtype: float64

pct_trades_trailing_activated distribution:
count    214.000000
mean      51.091589
std        8.395154
min       25.600000
25%       45.500000
50%       51.200000
75%       57.100000
max       77.800000
Name: pct_trades_trailing_activated, dtype: float64
